# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The notebook covers loading metadata, exploring record sets and fields by their `@id`, extracting data with pandas, running basic exploratory analysis, and visualization.

### Dataset Source

The dataset Croissant schema is at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values. Listing all record sets and fields to use their identifiers in subsequent steps.

In [ ]:
# Helper: print structure, referencing all entities by their @id
record_sets_info = []

print("Record Sets Available (by @id):\n------------------------------")
for rec_set in dataset.record_sets:
    print(f"@id: {rec_set['@id']}")
    rec_info = {'@id': rec_set['@id'], 'name': rec_set.get('name', '<no name>'), 'fields': []}
    print("  Fields:")
    for field in rec_set['fields']:
        fid = field['@id']
        fname = field.get('name', '<no name>')
        print(f"    @id: {fid} (name: {fname})")
        rec_info['fields'].append(fid)
    record_sets_info.append(rec_info)

# For later use:
record_set_ids = [r['@id'] for r in record_sets_info]

## 3. Data Extraction

Load data from each record set into a pandas DataFrame. All record sets are referenced by their `@id`. Field columns also use their `@id`, as required.

In [ ]:
# Extract and load data for all record sets, using @id for fields and columns
dataframes = {}
for rec_set_id in record_set_ids:
    records = list(dataset.records(record_set=rec_set_id))
    df = pd.DataFrame(records)
    dataframes[rec_set_id] = df
    print(f"\nColumns for record set '{rec_set_id}':")
    print(df.columns.tolist())

# Select a record set for EDA and preview
main_record_set_id = record_set_ids[0]
print(f"\nPreview for record set '{main_record_set_id}':")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps. We'll reference all fields and columns by their `@id`. Example: filtering for age > 50 and normalizing age and grouping by sex.

In [ ]:
# Identify a numeric field and grouping field by inspecting column @ids above.
# Example field IDs: change according to actual IDs printed above.

# Replace below with the appropriate @id for the 'Age' and 'Sex' fields as shown above
numeric_field = '<@id_for_age_field>'  # e.g. 'https://api.app.sen.science/frontiers/7862866/age'
group_field = '<@id_for_sex_field>'   # e.g. 'https://api.app.sen.science/frontiers/7862866/sex'
threshold = 50

df = dataframes[main_record_set_id]
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (referenced by @id):")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} values:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Grouping
    if group_field in df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field} (@id):")
        print(grouped)
else:
    print(f"Numeric field {numeric_field} not found in columns. Please check the field @id.")

## 5. Visualization

Visualize the distribution of the numeric field (e.g., age) and relationship to the grouping field (e.g., sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if field IDs were set correctly
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=15)
    plt.xlabel(numeric_field + ' (@id)')
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.show()

    # Boxplot by grouping field
    if group_field in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.xlabel(group_field + ' (@id)')
        plt.ylabel(numeric_field + ' (@id)')
        plt.title(f'{numeric_field} by {group_field} (@id)')
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, process, and visualize the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id`. By using schema-driven metadata, you can ensure reproducibility and clarity in aligning code with dataset semantics.

For further exploration, see the mlcroissant [documentation](https://github.com/mlcommons/croissant) for advanced processing and schema queries.